# Power Grid Voltage Control — DQN Reinforcement Learning

Trains a DQN agent to monitor power grid sensor data and recommend
the correct control action when a fault is detected.

**Run each cell in order. Total time: ~15 minutes.**

### Pipeline after training:
```
Edge Device → IoT Core → Lambda → SageMaker Endpoint → Device Shadow → Edge Device
```

## Cell 1 — Install dependencies

In [ ]:
!pip install torch numpy scikit-learn boto3 -q

## Cell 2 — DQN model architecture (5 → 128 → 64 → 5)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import json
import os
import tarfile
import time
from collections import deque
from sklearn.preprocessing import MinMaxScaler


class DQN(nn.Module):
    """
    Deep Q-Network.
    Input:  5 normalized grid features
    Hidden: 128 neurons → 64 neurons (ReLU activation)
    Output: 5 Q-values (one per action)
    """
    def __init__(self, input_dim=5, output_dim=5):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.out = nn.Linear(64, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.out(x)


class ReplayMemory:
    """Stores past (state, action, reward, next_state) tuples for batch training."""
    def __init__(self, capacity=50000):
        self.memory = deque(maxlen=capacity)
    def push(self, t):   self.memory.append(t)
    def sample(self, n): return random.sample(self.memory, n)
    def __len__(self):   return len(self.memory)


print('Architecture: 5 inputs → Linear(128) → ReLU → Linear(64) → ReLU → Linear(5 actions)')

## Cell 3 — Grid data generator

In [ ]:
def generate_grid_episode(n_steps=500, fault_prob=0.20):
    """
    Generate one episode of power grid sensor data.

    Features:
      [0] voltage_v:     grid voltage in Volts
      [1] current_a:     line current in Amperes
      [2] frequency_hz:  grid frequency in Hertz
      [3] temperature_c: transformer temperature in Celsius
      [4] power_factor:  power factor 0-1

    Normal ranges:
      V: 400-430V  I: 60-100A  Hz: 49.9-50.1  T: 30-55C  PF: 0.90-0.99

    Fault ranges:
      voltage_sag:        V = 340-385
      voltage_swell:      V = 440-470
      overcurrent:        I = 130-180
      frequency_drop:    Hz = 48.0-49.2
      overtemperature:    T = 70-95
      low_power_factor:  PF = 0.55-0.78
      multi_fault:       multiple simultaneously
    """
    FAULTS = [
        'voltage_sag', 'voltage_swell', 'overcurrent',
        'frequency_drop', 'overtemperature', 'low_power_factor', 'multi_fault'
    ]

    readings  = []
    labels    = []  # for tracking
    state     = 'normal'
    fault_rem = 0

    for t in range(n_steps):
        # state transition
        if fault_rem <= 0:
            state     = random.choice(FAULTS) if random.random() < fault_prob else 'normal'
            fault_rem = random.randint(5, 20) if state != 'normal' else 0
        else:
            fault_rem -= 1

        if state == 'normal':
            v, i, f, t_, pf = 415+random.gauss(0,3), 82+random.gauss(0,4), 50+random.gauss(0,0.03), 42+random.gauss(0,2), 0.95+random.gauss(0,0.01)
        elif state == 'voltage_sag':
            v, i, f, t_, pf = random.uniform(340,385), 82+random.gauss(0,4), 50+random.gauss(0,0.03), 42+random.gauss(0,2), 0.95+random.gauss(0,0.01)
        elif state == 'voltage_swell':
            v, i, f, t_, pf = random.uniform(440,470), 82+random.gauss(0,4), 50+random.gauss(0,0.03), 42+random.gauss(0,2), 0.95+random.gauss(0,0.01)
        elif state == 'overcurrent':
            v, i, f, t_, pf = 415+random.gauss(0,3), random.uniform(130,180), 50+random.gauss(0,0.03), 42+random.gauss(0,2), 0.95+random.gauss(0,0.01)
        elif state == 'frequency_drop':
            v, i, f, t_, pf = 415+random.gauss(0,3), 82+random.gauss(0,4), random.uniform(48.0,49.2), 42+random.gauss(0,2), 0.95+random.gauss(0,0.01)
        elif state == 'overtemperature':
            v, i, f, t_, pf = 415+random.gauss(0,3), 82+random.gauss(0,4), 50+random.gauss(0,0.03), random.uniform(70,95), 0.95+random.gauss(0,0.01)
        elif state == 'low_power_factor':
            v, i, f, t_, pf = 415+random.gauss(0,3), 82+random.gauss(0,4), 50+random.gauss(0,0.03), 42+random.gauss(0,2), random.uniform(0.55,0.78)
        elif state == 'multi_fault':
            v, i, f, t_, pf = random.uniform(340,385), random.uniform(130,180), random.uniform(48.0,49.5), random.uniform(65,95), random.uniform(0.55,0.78)

        readings.append([v, i, f, t_, max(0.0, min(1.0, pf))])
        labels.append(state)

    return np.array(readings, dtype=np.float32), labels


# quick preview
sample, sample_labels = generate_grid_episode(10, fault_prob=0.5)
print(f'Sample (V, I, Hz, T, PF):')
print(f'{"V":>8} {"I":>7} {"Hz":>7} {"T":>6} {"PF":>6}  State')
print('-' * 55)
for row, lbl in zip(sample, sample_labels):
    flag = '⚠' if lbl != 'normal' else ' '
    print(f'{flag}{row[0]:7.2f} {row[1]:7.2f} {row[2]:7.3f} {row[3]:6.1f} {row[4]:6.3f}  {lbl}')

## Cell 4 — Actions and reward function

In [ ]:
ACTION_NAMES = {
    0: 'no_action',          # grid is healthy — do nothing
    1: 'shed_load',          # overcurrent — disconnect non-critical loads
    2: 'switch_backup',      # voltage sag — switch to backup supply
    3: 'reduce_generation',  # voltage swell — reduce generation output
    4: 'alert_operator'      # frequency/temp/multi — human intervention needed
}


def detect_fault_type(state_raw):
    """
    Detect what type of fault is present from raw (unnormalized) values.
    Returns the ideal action for this fault.
    """
    v, i, f, t, pf = state_raw
    faults = []
    if v < 395:         faults.append('voltage_sag')
    if v > 435:         faults.append('voltage_swell')
    if i > 115:         faults.append('overcurrent')
    if f < 49.5:        faults.append('frequency_drop')
    if t > 65:          faults.append('overtemperature')
    if pf < 0.85:       faults.append('low_power_factor')

    if len(faults) > 1: return 4  # multi fault → alert operator
    if 'voltage_sag'  in faults: return 2  # switch backup
    if 'voltage_swell'in faults: return 3  # reduce generation
    if 'overcurrent'  in faults: return 1  # shed load
    if 'frequency_drop'  in faults: return 4  # alert operator
    if 'overtemperature' in faults: return 4  # alert operator
    if 'low_power_factor'in faults: return 4  # alert operator
    return 0  # normal — no action


def grid_reward(state_raw, action):
    """
    Reward function — teaches the agent what good behavior looks like.

    Scoring logic:
      +3.0  correct action during fault (agent identified and acted correctly)
      +2.0  no action during normal grid (agent correctly stayed idle)
      -3.0  wrong action during fault (agent picked wrong intervention)
      -3.0  no action during fault (agent ignored a real problem)
      -1.0  took action during normal grid (unnecessary intervention)

    Voltage scoring (always added):
      +1.0  voltage in normal range 400-430V
      -1.0  voltage outside normal range

    Frequency scoring (always added):
      +0.5  frequency in normal range 49.9-50.1 Hz
      -0.5  frequency outside normal range

    Temperature penalty:
      -1.0  transformer temperature above 65C
    """
    v, i, f, t, pf = state_raw

    # continuous scores — always applied
    v_score = 1.0  if 400 <= v <= 430 else -1.0
    f_score = 0.5  if 49.9 <= f <= 50.1 else -0.5
    t_score = -1.0 if t > 65 else 0.0

    # ideal action for this state
    ideal_action = detect_fault_type(state_raw)
    is_fault     = ideal_action > 0

    if is_fault and action == ideal_action:
        action_score = 3.0    # perfect — right action for this fault
    elif not is_fault and action == 0:
        action_score = 2.0    # correct — did nothing when all is well
    elif is_fault and action == 0:
        action_score = -3.0   # bad — ignored a real fault
    elif is_fault and action != ideal_action:
        action_score = -3.0   # bad — wrong intervention
    else:
        action_score = -1.0   # unnecessary intervention on normal grid

    return v_score + f_score + t_score + action_score


# test reward function
print('Reward function test:')
print(f'  Normal grid + no_action:       {grid_reward([415, 82, 50.0, 42, 0.95], 0):.1f}  (expected +3.5)')
print(f'  Voltage sag + switch_backup:   {grid_reward([360, 82, 50.0, 42, 0.95], 2):.1f}  (expected +3.0)')
print(f'  Voltage sag + no_action:       {grid_reward([360, 82, 50.0, 42, 0.95], 0):.1f}  (expected -4.0)')
print(f'  Overcurrent + shed_load:       {grid_reward([415, 150, 50.0, 42, 0.95], 1):.1f}  (expected +3.5)')
print(f'  Normal grid + shed_load:       {grid_reward([415, 82, 50.0, 42, 0.95], 1):.1f}  (expected +1.5)')

## Cell 5 — Train the DQN agent

In [ ]:
# ── hyperparameters ──────────────────────────────────────────
NUM_EPISODES  = 500    # number of simulation runs (increase for better model)
STEPS_PER_EP  = 1000   # sensor readings per episode
FAULT_PROB    = 0.20   # 20% chance of fault per step
BATCH_SIZE    = 64
GAMMA         = 0.99   # discount factor — how much future rewards matter
LR            = 1e-3   # learning rate
EPS_START     = 1.0    # start fully random
EPS_END       = 0.01   # end mostly learned
EPS_DECAY     = 0.997  # slower decay for more exploration
TARGET_UPDATE = 10     # sync target network every N episodes
MEMORY_SIZE   = 50000  # replay buffer size

# total training samples = NUM_EPISODES x STEPS_PER_EP = 500,000
print(f'Total training samples: {NUM_EPISODES * STEPS_PER_EP:,}')
print(f'Fault probability:      {FAULT_PROB*100:.0f}%')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device:        {device}')

# fit scaler on 5000 samples covering all fault ranges
scaler_data, _ = generate_grid_episode(5000, fault_prob=0.30)
scaler = MinMaxScaler()
scaler.fit(scaler_data)
print(f'Scaler fitted on 5,000 samples covering all fault ranges')

# initialize networks
policy_net = DQN(5, 5).to(device)
target_net = DQN(5, 5).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=LR)
memory    = ReplayMemory(MEMORY_SIZE)
eps       = EPS_START
all_rewards = []

print(f'\nStarting training: {NUM_EPISODES} episodes x {STEPS_PER_EP} steps')
print('─' * 65)

for ep in range(NUM_EPISODES):
    raw_data, _ = generate_grid_episode(STEPS_PER_EP, FAULT_PROB)
    normed      = scaler.transform(raw_data).astype(np.float32)

    total_reward = 0
    correct      = 0

    for t in range(len(normed) - 1):
        state     = normed[t]
        state_raw = raw_data[t]

        # epsilon-greedy action selection
        if random.random() < eps:
            action = random.randrange(5)
        else:
            with torch.no_grad():
                sv     = torch.tensor(state).unsqueeze(0).to(device)
                action = policy_net(sv).argmax().item()

        reward     = grid_reward(state_raw, action)
        next_state = normed[t + 1]
        total_reward += reward

        if action == detect_fault_type(state_raw):
            correct += 1

        memory.push((state, action, reward, next_state))

        # optimize
        if len(memory) >= BATCH_SIZE:
            batch = list(zip(*memory.sample(BATCH_SIZE)))
            sb  = torch.tensor(np.array(batch[0])).to(device)
            ab  = torch.tensor(batch[1]).unsqueeze(1).to(device)
            rb  = torch.tensor(batch[2], dtype=torch.float32).to(device)
            nsb = torch.tensor(np.array(batch[3])).to(device)

            q_vals     = policy_net(sb).gather(1, ab).squeeze()
            next_q     = target_net(nsb).max(1)[0].detach()
            expected_q = rb + GAMMA * next_q

            loss = nn.functional.mse_loss(q_vals, expected_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    eps = max(EPS_END, eps * EPS_DECAY)
    if ep % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    all_rewards.append(total_reward)
    accuracy = correct / STEPS_PER_EP * 100

    if (ep + 1) % 50 == 0 or ep == 0:
        avg = np.mean(all_rewards[-10:])
        print(f'  Ep {ep+1:4d}/{NUM_EPISODES} | reward={total_reward:9.1f} | avg10={avg:9.1f} | acc={accuracy:5.1f}% | eps={eps:.3f}')

print('─' * 65)
print(f'Training complete. Final accuracy: {correct/STEPS_PER_EP*100:.1f}%')

## Cell 6 — Save model files

In [ ]:
os.makedirs('model_output', exist_ok=True)

torch.save(policy_net.state_dict(), 'model_output/policy_net.pth')

scaler_params = {
    'min':   scaler.data_min_.tolist(),
    'max':   scaler.data_max_.tolist(),
    'scale': scaler.scale_.tolist(),
}
with open('model_output/scaler_params.json', 'w') as f:
    json.dump(scaler_params, f)

with open('model_output/action_names.json', 'w') as f:
    json.dump(ACTION_NAMES, f)

# save feature metadata
features = {
    'names':  ['voltage_v', 'current_a', 'frequency_hz', 'temperature_c', 'power_factor'],
    'normal': {'voltage_v': [400,430], 'current_a': [60,100], 'frequency_hz': [49.9,50.1], 'temperature_c': [30,55], 'power_factor': [0.90,0.99]},
    'fault':  {'voltage_sag': 'V<395', 'voltage_swell': 'V>435', 'overcurrent': 'I>115', 'frequency_drop': 'Hz<49.5', 'overtemperature': 'T>65', 'low_power_factor': 'PF<0.85'}
}
with open('model_output/features.json', 'w') as f:
    json.dump(features, f)

print('Saved:')
print('  model_output/policy_net.pth')
print('  model_output/scaler_params.json')
print('  model_output/action_names.json')
print('  model_output/features.json')

## Cell 7 — Write inference.py

In [ ]:
%%writefile model_output/inference.py

import torch
import torch.nn as nn
import numpy as np
import json
import os


class DQN(nn.Module):
    def __init__(self, input_dim=5, output_dim=5):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.out = nn.Linear(64, output_dim)
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.out(x)


ACTION_NAMES = {
    0: 'no_action', 1: 'shed_load', 2: 'switch_backup',
    3: 'reduce_generation', 4: 'alert_operator'
}

scaler_params = None


def model_fn(model_dir):
    global scaler_params
    with open(os.path.join(model_dir, 'scaler_params.json')) as f:
        scaler_params = json.load(f)
    net = DQN(5, 5)
    net.load_state_dict(torch.load(
        os.path.join(model_dir, 'policy_net.pth'), map_location='cpu'
    ))
    net.eval()
    return net


def input_fn(request_body, content_type):
    data = json.loads(request_body)
    raw = [
        float(data.get('voltage_v',     415.0)),
        float(data.get('current_a',      82.0)),
        float(data.get('frequency_hz',   50.0)),
        float(data.get('temperature_c',  42.0)),
        float(data.get('power_factor',   0.95)),
    ]
    mn = np.array(scaler_params['min'])
    sc = np.array(scaler_params['scale'])
    normed = (np.array(raw) - mn) * sc
    return torch.tensor(normed, dtype=torch.float32).unsqueeze(0)


def predict_fn(input_tensor, model):
    with torch.no_grad():
        q_values = model(input_tensor)
    action     = q_values.argmax(dim=1).item()
    confidence = torch.softmax(q_values, dim=1).max().item()
    return {
        'action':      action,
        'action_name': ACTION_NAMES.get(action, 'unknown'),
        'confidence':  round(confidence, 4),
        'q_values':    q_values.squeeze().tolist(),
    }


def output_fn(prediction, accept):
    return json.dumps(prediction), 'application/json'

## Cell 8 — Package model.tar.gz

In [ ]:
with tarfile.open('model.tar.gz', 'w:gz') as tar:
    for fname in ['policy_net.pth', 'scaler_params.json', 'action_names.json', 'features.json', 'inference.py']:
        tar.add(f'model_output/{fname}', arcname=fname)

with tarfile.open('model.tar.gz', 'r:gz') as tar:
    print('model.tar.gz contents:')
    for name in tar.getnames():
        print(f'  {name}')

## Cell 9 — Upload to S3

In [ ]:
import boto3

BUCKET = 'grid-dev-data-lake'
REGION = 'us-east-1'
ROLE   = 'arn:aws:iam::ACCOUNT_ID_REDACTED:role/grid-dev-sagemaker-role'

s3 = boto3.client('s3', region_name=REGION)
s3.upload_file('model.tar.gz', BUCKET, 'grid-voltage-rl/model/model.tar.gz')

s3_model_path = f's3://{BUCKET}/grid-voltage-rl/model/model.tar.gz'
print(f'Uploaded to: {s3_model_path}')

## Cell 10 — Deploy SageMaker endpoint

In [ ]:
import boto3, time

sm            = boto3.client('sagemaker', region_name='us-east-1')
ENDPOINT_NAME = 'grid-voltage-rl-v1'
MODEL_NAME    = 'grid-voltage-model'
CONFIG_NAME   = 'grid-voltage-config'
IMAGE         = '763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:2.0-cpu-py310'

for fn, kw in [(sm.delete_endpoint, {'EndpointName': ENDPOINT_NAME}),
               (sm.delete_endpoint_config, {'EndpointConfigName': CONFIG_NAME}),
               (sm.delete_model, {'ModelName': MODEL_NAME})]:
    try: fn(**kw)
    except: pass

sm.create_model(
    ModelName=MODEL_NAME, ExecutionRoleArn=ROLE,
    PrimaryContainer={
        'Image': IMAGE, 'ModelDataUrl': s3_model_path,
        'Environment': {'SAGEMAKER_PROGRAM': 'inference.py', 'SAGEMAKER_SUBMIT_DIRECTORY': s3_model_path}
    }
)
sm.create_endpoint_config(
    EndpointConfigName=CONFIG_NAME,
    ProductionVariants=[{'VariantName': 'main', 'ModelName': MODEL_NAME, 'InstanceType': 'ml.t2.medium', 'InitialInstanceCount': 1}]
)
sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=CONFIG_NAME)
print('Creating endpoint... (6-10 min)')

while True:
    s = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)['EndpointStatus']
    print(f'  {s}')
    if s in ('InService', 'Failed'): break
    time.sleep(30)

print(f'Endpoint: {ENDPOINT_NAME}')

## Cell 11 — Test: normal grid (expect no_action)

In [ ]:
runtime       = boto3.client('sagemaker-runtime', region_name='us-east-1')
ENDPOINT_NAME = 'grid-voltage-rl-v1'

normal = {'voltage_v': 415.3, 'current_a': 82.7, 'frequency_hz': 49.98, 'temperature_c': 42.1, 'power_factor': 0.94}
resp   = runtime.invoke_endpoint(EndpointName=ENDPOINT_NAME, ContentType='application/json', Body=json.dumps(normal))
r      = json.loads(resp['Body'].read())
print(f'Normal grid → action={r["action"]} ({r["action_name"]}) confidence={r["confidence"]}')
print(f'Expected: action=0 (no_action)')

## Cell 12 — Test all fault types

In [ ]:
tests = [
    ('Normal',          {'voltage_v':415, 'current_a':82,  'frequency_hz':50.0,  'temperature_c':42, 'power_factor':0.95}, 'no_action'),
    ('Voltage sag',     {'voltage_v':355, 'current_a':82,  'frequency_hz':50.0,  'temperature_c':42, 'power_factor':0.95}, 'switch_backup'),
    ('Voltage swell',   {'voltage_v':460, 'current_a':82,  'frequency_hz':50.0,  'temperature_c':42, 'power_factor':0.95}, 'reduce_generation'),
    ('Overcurrent',     {'voltage_v':415, 'current_a':160, 'frequency_hz':50.0,  'temperature_c':42, 'power_factor':0.95}, 'shed_load'),
    ('Frequency drop',  {'voltage_v':415, 'current_a':82,  'frequency_hz':48.5,  'temperature_c':42, 'power_factor':0.95}, 'alert_operator'),
    ('Overtemperature', {'voltage_v':415, 'current_a':82,  'frequency_hz':50.0,  'temperature_c':85, 'power_factor':0.95}, 'alert_operator'),
    ('Low PF',          {'voltage_v':415, 'current_a':82,  'frequency_hz':50.0,  'temperature_c':42, 'power_factor':0.62}, 'alert_operator'),
    ('Multi fault',     {'voltage_v':355, 'current_a':160, 'frequency_hz':48.5,  'temperature_c':85, 'power_factor':0.62}, 'alert_operator'),
]

print(f'{"Scenario":<20} {"Expected":<20} {"Got":<20} {"Conf":>6} {"Match"}')
print('-' * 75)

correct = 0
for name, payload, expected in tests:
    resp = runtime.invoke_endpoint(EndpointName=ENDPOINT_NAME, ContentType='application/json', Body=json.dumps(payload))
    r    = json.loads(resp['Body'].read())
    match = '✅' if r['action_name'] == expected else '❌'
    if r['action_name'] == expected: correct += 1
    print(f'{name:<20} {expected:<20} {r["action_name"]:<20} {r["confidence"]:6.4f} {match}')

print(f'\nAccuracy: {correct}/{len(tests)} = {correct/len(tests)*100:.0f}%')

## Cell 13 — Delete endpoint (run when done)
⚠️ Costs $0.065/hour while running

In [ ]:
# UNCOMMENT TO DELETE
# sm.delete_endpoint(EndpointName='grid-voltage-rl-v1')
# print('Endpoint deleted')